In [0]:
import json
from pyspark.sql import functions as F
from pyspark.ml.feature import StringIndexerModel, VectorAssembler
from pyspark.ml.regression import RandomForestRegressionModel
from datetime import datetime

dbutils.widgets.text("pulocationid", "")
dbutils.widgets.text("dolocationid", "")
dbutils.widgets.text("hour", "")

pu_raw = dbutils.widgets.get("pulocationid")
do_raw = dbutils.widgets.get("dolocationid")
hour_raw = dbutils.widgets.get("hour")

if pu_raw == "" or do_raw == "":
    raise ValueError("pulocationid and dolocationid are required")

pu_id = int(pu_raw)
do_id = int(do_raw)
hour = int(hour_raw) if hour_raw.strip() != "" else datetime.now().hour

if not (0 <= hour <= 23):
    raise ValueError("hour must be between 0 and 23")

time_group = hour // 4

base_path = "dbfs:/FileStore/taxi_models"

indexer_model = StringIndexerModel.load(f"{base_path}/service_type_indexer")
model_time = RandomForestRegressionModel.load(f"{base_path}/trip_time_model")
model_cost = RandomForestRegressionModel.load(f"{base_path}/total_amount_model")

assembler = VectorAssembler(
    inputCols=["pulocationid", "dolocationid", "service_type_idx", "time_group"],
    outputCol="features"
)

service_types = ["Yellow", "Green", "Uber", "Lyft"]

input_rows = [(pu_id, do_id, s, time_group) for s in service_types]

input_df = spark.createDataFrame(
    input_rows,
    ["pulocationid", "dolocationid", "service_type", "time_group"]
)

input_df = indexer_model.transform(input_df)
input_df = assembler.transform(input_df)

pred_time = model_time.transform(input_df).select(
    "pulocationid", "dolocationid", "service_type", "time_group",
    F.col("prediction").alias("predicted_trip_time")
)

pred_cost = model_cost.transform(input_df).select(
    "pulocationid", "dolocationid", "service_type", "time_group",
    F.col("prediction").alias("predicted_total_amount")
)

result_df = (
    pred_time.join(
        pred_cost,
        on=["pulocationid", "dolocationid", "service_type", "time_group"],
        how="inner"
    )
    .orderBy("predicted_total_amount")
)

result = [row.asDict() for row in result_df.collect()]

payload = {
    "pulocationid": pu_id,
    "dolocationid": do_id,
    "hour": hour,
    "time_group": time_group,
    "predictions": result
}

dbutils.notebook.exit(json.dumps(payload))